# XGBoost Model

Reads the dataframe made by `build_dataframe.ipynb` and trains an XGBoost model that gives each 10-K or 10-Q a probability of a bankruptcy 8-K within 365 days. The data is split by company and by time, the class imbalance is handled with class weights, and Optuna tunes the settings inside each round. The last section saves the files the investigator agent reads (`agent/shared/`).

In [1]:
#pip install xgboost optuna scikit-learn
import os

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 40

## Loading the Dataframe

In [2]:
#this is the dataframe made by build_dataframe.ipynb
everything = pd.read_csv("model_dataframe_2023_Q1_to_2026_Q3.txt", sep="\t", low_memory=False, parse_dates=["filed_date"])

#I keep the rows where we know the answer (filed at least a year before the labels were made) and the
#company wasn't already bankrupt
df = everything[everything["label_known"] & ~everything["already_bankrupt"]].reset_index(drop=True)
print(df.shape)
print(df["Bankruptcy"].value_counts())
print("companies:", df["cik"].nunique(), " companies with a 1:", df.loc[df["Bankruptcy"] == 1, "cik"].nunique())

#only the 8 variables we agreed on. Empty values stay empty, XGBoost handles them
features = ["total_assets", "current_assets", "current_liabilities", "total_liabilities",
            "revenue", "net_income", "operating_income", "operating_cash_flow"]

(64537, 23)
Bankruptcy
0.0    63928
1.0      609
Name: count, dtype: int64
companies: 7137  companies with a 1: 204


## Model and Tuning

In [3]:
#only about 1% of rows are 1, so the 1s get more weight (scale_pos_weight): the model pays more for missing a 1.
#Optuna picks how much, together with the other settings
def make_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 30, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        #the xgboost docs say max_delta_step helps logistic regression when the classes are very imbalanced
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 150, log=True),
    }


def new_model(params):
    return XGBClassifier(**params, tree_method="hist", eval_metric="aucpr", n_jobs=8, random_state=42)


def shuffle_rows(X, y, seed=42):
    keep = np.random.default_rng(seed).permutation(np.arange(len(y)))
    return X.iloc[keep], y.iloc[keep]


def cv_scores(params, data, folds=4):
    #4 fold cross validation inside the training part, split by company (train on 3 groups, validate on 1).
    #Returns the PR-AUC of each fold, which is what Optuna maximizes
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=7)
    fold_pr_auc = []
    for fit_index, val_index in cv.split(data, data["Bankruptcy"], groups=data["cik"]):
        X_fit, y_fit = shuffle_rows(data.loc[fit_index, features], data.loc[fit_index, "Bankruptcy"])
        scores = new_model(params).fit(X_fit, y_fit).predict_proba(data.loc[val_index, features])[:, 1]
        fold_pr_auc.append(average_precision_score(data.loc[val_index, "Bankruptcy"], scores))
        yield fold_pr_auc


def tune(data):
    def objective(trial):
        params = make_params(trial)
        for step, fold_pr_auc in enumerate(cv_scores(params, data)):
            #stop bad trials early
            trial.report(np.mean(fold_pr_auc), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return np.mean(fold_pr_auc)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1))
    study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params

## Split by Company and by Time

1. Every company is put in one of 4 groups (A, B, C, D) and stays there.
2. The filings are put in time order in half-year blocks.
3. 4 rounds: train on the blocks before, test on the next block, only on one group of companies. That group is left out of the training, so the model never saw the companies it is tested on (and the same bankruptcy can't be in train and test). Optuna tunes inside the training part only.

In [4]:
block_edges = pd.to_datetime(["2023-01-01", "2023-07-01", "2024-01-01", "2024-07-01", "2025-01-01", "2026-01-01"])
block_names = ["2023 H1", "2023 H2", "2024 H1", "2024 H2", "2025"]
block = pd.cut(df["filed_date"], block_edges, right=False, labels=False).values

company_group = np.zeros(len(df), int)
for g, (_, group_index) in enumerate(StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42).split(
        df, df["Bankruptcy"], groups=df["cik"])):
    company_group[group_index] = g

rounds, test_scores, importances = [], [], []
for fold in range(1, 5):
    fit_data = df[(block < fold) & (company_group != fold - 1)].reset_index(drop=True)
    test_data = df[(block == fold) & (company_group == fold - 1)]

    params = tune(fit_data)
    X_fit, y_fit = shuffle_rows(fit_data[features], fit_data["Bankruptcy"])
    model = new_model(params).fit(X_fit, y_fit)
    scores = model.predict_proba(test_data[features])[:, 1]
    y = test_data["Bankruptcy"].values

    test_scores.append(pd.Series(scores, index=test_data.index))
    #which features the model used the most (gain = how much each one improves the splits)
    gain = pd.Series(model.get_booster().get_score(importance_type="gain")).reindex(features).fillna(0)
    importances.append(gain / gain.sum())
    rounds.append({"round": fold, "train on": f"2023 H1 to {block_names[fold - 1]}", "test on": block_names[fold],
                   "train filings": len(fit_data), "train ones": int(fit_data["Bankruptcy"].sum()),
                   "train % of all": round(100 * len(fit_data) / len(df), 1),
                   "test filings": len(test_data), "test ones": int(y.sum()),
                   "test % of all": round(100 * len(test_data) / len(df), 1),
                   "ROC-AUC": roc_auc_score(y, scores)})

rounds = pd.DataFrame(rounds).set_index("round")
print("average ROC-AUC over the 4 rounds:", round(rounds["ROC-AUC"].mean(), 3),
      " spread (std):", round(rounds["ROC-AUC"].std(), 3))
rounds.round(3)

average ROC-AUC over the 4 rounds: 0.896  spread (std): 0.053


,train on,test on,train filings,train ones,train % of all,test filings,test ones,test % of all,ROC-AUC
round,,,,,,,,,
1,2023 H1 to 2023 H1,2023 H2,9415,136,14.6,3012,21,4.7,0.947
2,2023 H1 to 2023 H2,2024 H1,18621,232,28.9,2817,21,4.4,0.880
3,2023 H1 to 2024 H1,2024 H2,27167,292,42.1,2875,22,4.5,0.927
4,2023 H1 to 2024 H2,2025,35817,378,55.5,4180,21,6.5,0.828


## Feature Importance

In [5]:
#the share of the gain of each feature, averaged over the 4 round models
pd.concat(importances, axis=1).mean(axis=1).sort_values(ascending=False).round(3)

net_income             0.237
operating_income       0.205
operating_cash_flow    0.110
current_assets         0.099
current_liabilities    0.098
total_liabilities      0.095
total_assets           0.082
revenue                0.074
dtype: float64

## Saving the Data for the Investigator Agent

In [6]:
#the agent gets two files:
#  financials.csv: every report without the label columns, so it can't see who went bankrupt
#  risk_scores.csv: the model's probability for the test reports of the 4 rounds, from a model that never
#  saw that company or anything filed after the training blocks
os.makedirs("agent/shared", exist_ok=True)

shared_columns = ["adsh", "cik", "name", "sic", "form", "fp", "ddate", "filed"] + features
everything[shared_columns].to_csv("agent/shared/financials.csv", index=False)

risk_scores = df.loc[pd.concat(test_scores).index, ["adsh", "cik", "name", "form", "ddate", "filed"]].assign(
    probability=pd.concat(test_scores), scored_by="split by company and time")
risk_scores.to_csv("agent/shared/risk_scores.csv", index=False)
print("financials.csv:", len(everything), "reports   risk_scores.csv:", len(risk_scores), "reports")

financials.csv: 86927 reports   risk_scores.csv: 12884 reports
